# DEQ Export Demo — `lightstim.deq`

**[DEMO]** — the exporter is packaged in `lightstim/deq/`; this notebook only imports and demonstrates it.

Converts LightStim circuits into Microsoft's `.deq` DSL (from the open-source
[`qdk-ec`](https://github.com/microsoft/qdk-ec) toolkit). See the Light-DEQ plan for
background: this exporter replaces a from-scratch `.deq` generator that used to live in
`resource-superstaq/resource_estimation/ftqc/microarchitecture/surface-code-deq`, whose
generator was deleted but whose output (`generated/rotated_surface_code_d3.deq`) was kept
as a reference target.

This notebook:
1. Exports a small repetition-code memory circuit and checks the `.deq` text structurally.
2. Exports LightStim's own rotated surface code (d=3) and compares it against the
   resource-superstaq reference file's `CODE` parameters.
3. Notes the current scope/limitations.


In [1]:
import sys
from pathlib import Path

ROOT = Path("../..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lightstim.protocols.memory import MemoryExperiment
from lightstim.qec_code.repetition.repetition import RepetitionCode
from lightstim.qec_code.surface_code.rotated import (
    RotatedSurfaceCode,
    RotatedSurfaceCodeExtractionBlock,
)
from lightstim.deq import export_deq
from lightstim.deq.validate import deq_available, validate_deq_text

print("deq/deqagram grammar available for validation:", deq_available())


deq/deqagram grammar available for validation: True


## 1. Repetition code (d=3) — exact structural check

The exported circuit must be **noiseless**: pass `noise_params=None` to `MemoryExperiment`.

In [2]:
rep_patch = RepetitionCode(distance=3)
rep_exp = MemoryExperiment(qec_patch=rep_patch, rounds=3, noise_params=None, basis="Z")
rep_circuit = rep_exp.build()

print(f"Qubits: {rep_circuit.num_qubits}  Detectors: {rep_circuit.num_detectors}  "
      f"Observables: {rep_circuit.num_observables}")

rep_deq_text = export_deq(rep_exp.system, rep_circuit, gadget_name="Repetition")
print(rep_deq_text)


Qubits: 5  Detectors: 8  Observables: 1
# Generated by lightstim.deq.export; do not edit by hand.

CODE memory [[3,1,3]] {
    LOGICAL X0*X1*X2 Z0
    STABILIZER Z0*Z1
    STABILIZER Z1*Z2
}

GADGET Repetition {
    R 0 2 4 1 3
    TICK[SE_start]
    TICK
    CX 0 1 2 3
    TICK
    CX 2 1 4 3
    TICK
    TICK
    M 1 3
    DETECTOR rec[-2]
    DETECTOR rec[-1]
    TICK
    R 1 3
    TICK[SE_start]
    TICK
    CX 0 1 2 3
    TICK
    CX 2 1 4 3
    TICK
    TICK
    M 1 3
    DETECTOR rec[-4] rec[-2]
    DETECTOR rec[-3] rec[-1]
    REPEAT 1 {
        TICK
        R 1 3
        TICK[SE_start]
        TICK
        CX 0 1 2 3
        TICK
        CX 2 1 4 3
        TICK
        TICK
        M 1 3
        DETECTOR rec[-4] rec[-2]
        DETECTOR rec[-3] rec[-1]
    }
    TICK
    M 0 2 4
    DETECTOR rec[-5] rec[-3] rec[-2]
    DETECTOR rec[-4] rec[-2] rec[-1]
    OBSERVABLE_INCLUDE rec[-3]
}



In [3]:
rep_parsed = validate_deq_text(rep_deq_text)
print("Parsed OK:", rep_parsed is not None)

if deq_available():
    from deq.circuit.model import CodeDefinition, GadgetDefinition
    code_def = next(d for d in rep_parsed.definitions if isinstance(d, CodeDefinition))
    gadget_def = next(d for d in rep_parsed.definitions if isinstance(d, GadgetDefinition))
    print(f"CODE {code_def.name}: n={code_def.n} k={code_def.k} d={code_def.d}, "
          f"{len(code_def.logicals)} logical(s), {len(code_def.stabilizers)} stabilizer(s)")
    print(f"GADGET {gadget_def.name}: {len(gadget_def.body)} body statements")
    assert (code_def.n, code_def.k, code_def.d) == (3, 1, 3)
    assert len(code_def.stabilizers) == 2  # d-1 Z-checks


Parsed OK: True
CODE memory: n=3 k=1 d=3, 1 logical(s), 2 stabilizer(s)
GADGET Repetition: 29 body statements


## 2. Rotated surface code (d=3) — primary Light-DEQ target

Structural comparison against
`resource-superstaq/resource_estimation/ftqc/microarchitecture/surface-code-deq/generated/rotated_surface_code_d3.deq`,
whose `CODE RotatedSurfaceCodeW3H3 [[9,1,3]]` block has 8 stabilizers and 1 logical pair.
Exact qubit-index equality isn't expected (different indexing schemes between LightStim's
IR and that project's from-scratch geometry code) — only the algebraic shape should match.


In [4]:
rsc_patch = RotatedSurfaceCode(distance=3)
rsc_exp = MemoryExperiment(
    qec_patch=rsc_patch,
    extraction_block_class=RotatedSurfaceCodeExtractionBlock,
    rounds=3,
    noise_params=None,
    basis="Z",
)
rsc_circuit = rsc_exp.build()

print(f"Qubits: {rsc_circuit.num_qubits}  Detectors: {rsc_circuit.num_detectors}  "
      f"Observables: {rsc_circuit.num_observables}")

rsc_deq_text = export_deq(rsc_exp.system, rsc_circuit, gadget_name="RotatedSurfaceMemory")
print(rsc_deq_text[:1500] + "\n... (truncated) ...")


Qubits: 17  Detectors: 24  Observables: 1
# Generated by lightstim.deq.export; do not edit by hand.

CODE memory [[9,1,3]] {
    LOGICAL X0*X3*X6 Z0*Z1*Z2
    STABILIZER X0*X1
    STABILIZER Z0*Z1*Z3*Z4
    STABILIZER Z2*Z5
    STABILIZER X1*X2*X4*X5
    STABILIZER Z3*Z6
    STABILIZER Z4*Z5*Z7*Z8
    STABILIZER X3*X4*X6*X7
    STABILIZER X7*X8
}

GADGET RotatedSurfaceMemory {
    R 1 2 3 7 8 9 13 14 15 0 4 5 6 10 11 12 16
    TICK[SE_start]
    H 0 6 12 16
    TICK
    CX 0 2 6 9 12 14 8 4 13 10 15 11
    TICK
    CX 0 1 6 8 12 13 2 4 7 10 9 11
    TICK
    CX 6 3 12 8 16 15 7 4 9 5 14 11
    TICK
    CX 6 2 12 7 16 14 1 4 3 5 8 11
    TICK
    H 0 6 12 16
    TICK
    M 0 4 5 6 10 11 12 16
    DETECTOR rec[-7]
    DETECTOR rec[-6]
    DETECTOR rec[-4]
    DETECTOR rec[-3]
    TICK
    R 0 4 5 6 10 11 12 16
    TICK[SE_start]
    H 0 6 12 16
    TICK
    CX 0 2 6 9 12 14 8 4 13 10 15 11
    TICK
    CX 0 1 6 8 12 13 2 4 7 10 9 11
    TICK
    CX 6 3 12 8 16 15 7 4 9 5 14 11
    TICK
 

In [5]:
rsc_parsed = validate_deq_text(rsc_deq_text)
print("Parsed OK:", rsc_parsed is not None)

if deq_available():
    from deq.circuit.model import CodeDefinition
    rsc_code_def = next(d for d in rsc_parsed.definitions if isinstance(d, CodeDefinition))
    print(f"CODE {rsc_code_def.name}: n={rsc_code_def.n} k={rsc_code_def.k} d={rsc_code_def.d}, "
          f"{len(rsc_code_def.logicals)} logical(s), {len(rsc_code_def.stabilizers)} stabilizer(s)")
    assert (rsc_code_def.n, rsc_code_def.k, rsc_code_def.d) == (9, 1, 3)
    assert len(rsc_code_def.stabilizers) == 8
    print("Matches resource-superstaq reference CODE RotatedSurfaceCodeW3H3 [[9,1,3]] "
          "(8 stabilizers, 1 logical pair).")


Parsed OK: True
CODE memory: n=9 k=1 d=3, 1 logical(s), 8 stabilizer(s)
Matches resource-superstaq reference CODE RotatedSurfaceCodeW3H3 [[9,1,3]] (8 stabilizers, 1 logical pair).


In [6]:
# Optional: parse the actual resource-superstaq reference file directly, if present
# on this machine, and print its CODE parameters side by side for comparison.
from deq.circuit import parser as _deq_parser
from deq.circuit.model import CodeDefinition as _CodeDefinition

ref_path = (
    ROOT.parent
    / "resource-superstaq/resource_estimation/ftqc/microarchitecture/surface-code-deq"
    / "generated/rotated_surface_code_d3.deq"
)
if deq_available() and ref_path.exists():
    ref_parsed = _deq_parser.parse(ref_path.read_text())
    ref_code_def = next(
        d for d in ref_parsed.definitions
        if isinstance(d, _CodeDefinition) and d.name == "RotatedSurfaceCodeW3H3"
    )
    print(f"reference: n={ref_code_def.n} k={ref_code_def.k} d={ref_code_def.d}, "
          f"{len(ref_code_def.logicals)} logical(s), {len(ref_code_def.stabilizers)} stabilizer(s)")
    print(f"lightstim: n={rsc_code_def.n} k={rsc_code_def.k} d={rsc_code_def.d}, "
          f"{len(rsc_code_def.logicals)} logical(s), {len(rsc_code_def.stabilizers)} stabilizer(s)")
else:
    print("Reference file or deq/deqagram not available in this environment; skipping direct comparison.")


reference: n=9 k=1 d=3, 1 logical(s), 8 stabilizer(s)
lightstim: n=9 k=1 d=3, 1 logical(s), 8 stabilizer(s)


## 3. Multi-patch export

`export_deq` supports any number of patches: each patch gets its own `CODE` block (with
its own local qubit numbering). Each patch also gets an `OUTPUT` declaration — but only
if its qubits still hold live quantum state at the end of the circuit (§3c explains why,
and how that was actually found).

### 3a. Two independent patches (no coupler) — transversal CNOT


In [7]:
from lightstim.protocols.cnot_trans import CNOTTransExperiment

cnot_exp = CNOTTransExperiment(
    code_patch_class=RotatedSurfaceCode,
    extraction_block_class=RotatedSurfaceCodeExtractionBlock,
    code_params_control={"distance": 3},
    rounds_before=1,
    rounds_after=1,
    noise_params=None,
)
cnot_circuit = cnot_exp.build()
print("patches:", list(cnot_exp.system.patches.keys()))

cnot_deq_text = export_deq(cnot_exp.system, cnot_circuit, gadget_name="CNOTTrans")
cnot_parsed = validate_deq_text(cnot_deq_text)
print("Parsed OK:", cnot_parsed is not None)

if deq_available():
    from deq.circuit.model import CodeDefinition, GadgetDefinition
    codes = {d.name: d for d in cnot_parsed.definitions if isinstance(d, CodeDefinition)}
    gadget = next(d for d in cnot_parsed.definitions if isinstance(d, GadgetDefinition))
    for name, c in codes.items():
        print(f"CODE {name}: n={c.n} k={c.k} d={c.d}, {len(c.stabilizers)} stabilizer(s)")
    print(f"GADGET {gadget.name}: {len(gadget.output_ports)} OUTPUT port(s)")
    assert set(codes) == {"control", "target"}

# 0 OUTPUT ports is CORRECT here: CNOTTransExperiment.build() ends in a full destructive
# data-qubit readout, so no live code instance survives for either patch -- see §3c.


Creating patches...
Setting up QEC system...
Writing coordinates...
Initializing data qubits...
Building 1 rounds of syndrome extraction (before CNOT)...
Applying transversal CNOT gate...
Building 1 rounds of syndrome extraction (after CNOT)...
Measuring data qubits...
patches: ['control', 'target']
Parsed OK: True
CODE control: n=9 k=1 d=3, 8 stabilizer(s)
CODE target: n=9 k=1 d=3, 8 stabilizer(s)
GADGET CNOTTrans: 0 OUTPUT port(s)


### 3b. Lattice surgery — coupler patches get their own CODE block

`TwoPatchLSExperiment` registers a *coupler* patch (merge/split ancilla). Each patch's
`CODE` block is built from its own canonical, per-patch `stabilizers`/`logical_ops`
(globalized to the physical circuit's qubit indices), not from the system's evolved
logical frame — that distinction matters here because entangling operations (transversal
gates, lattice surgery) update the system-level frame to track logical-operator
propagation for decoding, which is the wrong thing for a fixed `CODE` declaration. The
coupler gets a `k=0` stabilizer-state code, the same way the resource-superstaq
reference file represents its own mediator patch as `YBoundaryStateD3 [[8,0]]`.


In [8]:
import contextlib
import io

from lightstim.protocols.two_patch_ls import TwoPatchLSExperiment

with contextlib.redirect_stdout(io.StringIO()):
    ls_exp = TwoPatchLSExperiment(
        patch1_config={"distance": 3},
        patch2_config={"distance": 3},
        offset=(0, 10),
        interaction_type="ZZ",
        initial_state_patch1="X",
        initial_state_patch2="Z",
        measure_state_patch1="X",
        measure_state_patch2="Z",
        rounds=1,
        noise_params=None,
    )
    ls_circuit = ls_exp.build()

print("patches:", list(ls_exp.system.patches.keys()))
print("coupler_patches:", list(ls_exp.system.coupler_patches.keys()))

ls_deq_text = export_deq(ls_exp.system, ls_circuit, gadget_name="TwoPatchLS")
ls_parsed = validate_deq_text(ls_deq_text)
print("Parsed OK:", ls_parsed is not None)

if deq_available():
    from deq.circuit.model import CodeDefinition
    ls_codes = {d.name: d for d in ls_parsed.definitions if isinstance(d, CodeDefinition)}
    for name, c in ls_codes.items():
        print(f"CODE {name}: n={c.n} k={c.k} d={c.d}, {len(c.logicals)} logical(s), "
              f"{len(c.stabilizers)} stabilizer(s)")
    assert set(ls_codes) == set(ls_exp.system.patches)
    coupler_name = next(iter(ls_exp.system.coupler_patches))
    assert ls_codes[coupler_name].k == 0
    print(f"Coupler '{coupler_name}' correctly exported as a k=0 stabilizer-state code.")


patches: ['surface_code_1', 'surface_code_2', 'coupler_1_2']
coupler_patches: ['coupler_1_2']
Parsed OK: True
CODE surface_code_1: n=13 k=1 d=3, 1 logical(s), 12 stabilizer(s)
CODE surface_code_2: n=13 k=1 d=3, 1 logical(s), 12 stabilizer(s)
CODE coupler_1_2: n=22 k=0 d=None, 0 logical(s), 17 stabilizer(s)
Coupler 'coupler_1_2' correctly exported as a k=0 stabilizer-state code.


### 3c. `OUTPUT` is omitted after a terminal (destructive) readout

Found for real, not just reasoned about: an earlier version of this exporter *always*
emitted `OUTPUT` for every patch. Running that output for an H6-distillation circuit
(which ends in `MX` on all its data qubits) through **Bloqade Studio's real `deq` compile
step** — something this notebook can't do locally (no Rust toolchain here, so validation
below is grammar-only via `deqagram.parse`) — failed with:

```
GADGET 'H6DistillationLevel1Proxy' is invalid: the following output stabilizer(s) cannot
be expressed as a linear combination of input-virtual and internal measurements, so they
cannot be checked by the gadget's outcome code: ...
```

`deq`'s compiler verifies every declared `OUTPUT` stabilizer is reconstructible from the
gadget's own measurements ("automatic check discovery") — and correctly rejects an
`OUTPUT` over qubits that were just destructively measured, since there's no code left.
Fixed by detecting exactly that case (`_destructively_measured_qubits` in `export.py`):
a plain `M`/`MX`/`MY`/`MZ` with nothing after it consumes the qubit, so no `OUTPUT` is
emitted for it. This is also why the two-patch export above correctly has 0 output ports.


In [9]:
from lightstim.ir.builder import CircuitBuilder
from lightstim.ir.qec_system import QECSystem
from lightstim.ir.tracker import SyndromeTracker
from lightstim.qec_code.repetition.SE_block import RepetitionCodeExtractionBlock

# A circuit that prepares + runs one SE round but does NOT measure the data
# qubits out: the code instance survives, so OUTPUT should be present.
system = QECSystem()
system.add_patch(RepetitionCode(distance=3), name="memory")
tracker = SyndromeTracker(system.num_qubits, expected_num_logicals=system.num_logicals)
builder = CircuitBuilder(tracker, system, if_detector=True)
builder.write_coordinates()
data = sorted(system.data_indices)
builder.initialize({q: "Z" for q in data}, n=system.num_qubits)
builder.apply_syndrome_extraction(
    circuit_chunk=RepetitionCodeExtractionBlock(system).circuit, rounds=1
)
surviving_text = export_deq(system, builder.circuit, gadget_name="PrepareAndExtract")
print("code survives to the end -> OUTPUT present:")
print([line for line in surviving_text.splitlines() if "OUTPUT" in line])

print("\nfull memory experiment (ends in MZ) -> OUTPUT absent:")
print([line for line in rep_deq_text.splitlines() if "OUTPUT" in line] or ["(none)"])


code survives to the end -> OUTPUT present:
['    OUTPUT memory 0 2 4']

full memory experiment (ends in MZ) -> OUTPUT absent:
['(none)']


### 3d. Noisy circuits export fine

Stim's noise channels (`X_ERROR`, `DEPOLARIZE1`/`DEPOLARIZE2`, ...) aren't reserved
GADGET keywords in the `.deq` grammar, so they pass through as ordinary instructions —
this is unrelated to deq's own native `ERROR(p) <target>` statement (which only takes
CHECK/READOUT/LOGICAL targets and is never emitted here).


In [10]:
from lightstim.noise.config import NoiseConfig

noise = NoiseConfig(p_1q=1e-3, p_2q=1e-3, p_meas=1e-3, p_reset=1e-3)
noisy_exp = MemoryExperiment(qec_patch=RepetitionCode(distance=3), rounds=2, noise_params=noise, basis="Z")
noisy_circuit = noisy_exp.build()

noisy_deq_text = export_deq(noisy_exp.system, noisy_circuit, gadget_name="NoisyRepetition")
print(noisy_deq_text)

noisy_parsed = validate_deq_text(noisy_deq_text)
print("Parsed OK:", noisy_parsed is not None)


# Generated by lightstim.deq.export; do not edit by hand.

CODE memory [[3,1,3]] {
    LOGICAL X0*X1*X2 Z0
    STABILIZER Z0*Z1
    STABILIZER Z1*Z2
}

GADGET NoisyRepetition {
    R 0 2 4 1 3
    X_ERROR(0.001) 0 2 4 1 3
    TICK[SE_start]
    TICK
    CX 0 1 2 3
    DEPOLARIZE2(0.001) 0 1 2 3
    TICK
    CX 2 1 4 3
    DEPOLARIZE2(0.001) 2 1 4 3
    TICK
    TICK
    X_ERROR(0.001) 1 3
    M 1 3
    DETECTOR rec[-2]
    DETECTOR rec[-1]
    TICK
    R 1 3
    X_ERROR(0.001) 1 3
    TICK[SE_start]
    TICK
    CX 0 1 2 3
    DEPOLARIZE2(0.001) 0 1 2 3
    TICK
    CX 2 1 4 3
    DEPOLARIZE2(0.001) 2 1 4 3
    TICK
    TICK
    X_ERROR(0.001) 1 3
    M 1 3
    DETECTOR rec[-4] rec[-2]
    DETECTOR rec[-3] rec[-1]
    TICK
    X_ERROR(0.001) 0 2 4
    M 0 2 4
    DETECTOR rec[-5] rec[-3] rec[-2]
    DETECTOR rec[-4] rec[-2] rec[-1]
    OBSERVABLE_INCLUDE rec[-3]
}

Parsed OK: True


## 4. Scope & limitations (v1)

- **Multi-patch export works, including coupler patches** (one `CODE` per patch, in a
  single GADGET — see §3a/§3b). Each patch's CODE block comes from its own canonical,
  per-patch stabilizers/logicals (correctly globalized to the physical circuit's qubit
  indices), not the system's Heisenberg-evolved logical frame — using the latter by
  mistake once inflated a two-patch transversal-CNOT export's second patch from
  `[[9,1,3]]` to a bogus `[[18,1,3]]`.
- **`OUTPUT` is emitted only when a patch's qubits survive to the end of the circuit**
  (§3c) — found via a real `deq` compile failure in Bloqade Studio, not just reasoned
  about locally.
- **Noisy circuits export fine** (§3d) — Stim's noise channels aren't reserved GADGET
  keywords, so they pass through as ordinary instructions.
- **One monolithic GADGET per circuit**, not split into Prepare/SyndromeExtraction/Measure
  sub-gadgets wired via `COMPOSE` (unlike the resource-superstaq reference file). Splitting
  naively breaks `rec[-k]` scoping, since deq scopes measurement records *per GADGET* and a
  DETECTOR comparing against a prior gadget's measurement would go out of range. Crossing a
  gadget boundary correctly needs deq's `IN<p>.S<s>` virtual-port syntax — and deq's own
  "automatic check discovery" compiler pass is presumably involved in how that's meant to
  be used, which can only be verified against a working `deq transpile` or a real Studio
  compile (grammar-level parsing alone can't confirm semantic correctness, as §3c's finding
  underlines). Left as follow-up rather than shipped as an unverified guess.
- **PPVM-backed simulation is not yet implemented** — blocked on a Rust toolchain issue in
  this environment; see the Light-DEQ plan for the intended design
  (`lightstim/simulation/ppvm_backend/`).
